# 11 · Is the difference real? Hypothesis tests and confidence intervals
Question: **do weekends earn more per day than weekdays?** A pattern in a chart is not proof. Statistics tells you how surprised you should be if there were really no difference.

In [ ]:
import sys
sys.path.insert(0, "..")          # lets the notebook import common.py from python_practice/
from common import *              # load_operations, check, OUT, ...
import numpy as np
import pandas as pd

df = load_operations()            # reads the dashboard API if it is running, else data_samples/operations_clean.csv
df["not_completed"] = (df["status"] != "Completed").astype(int)
from scipy import stats

## Choose the unit of analysis first
The unit is the thing you can treat as one independent observation. Revenue per **record** has 3,650 rows, but records on the same day share conditions. The honest unit here is a **day** (total revenue per day).

In [ ]:
daily = df.groupby(["record_date", "is_weekend"])["revenue"].sum().reset_index()
weekend = daily.loc[daily["is_weekend"], "revenue"]
weekday = daily.loc[~daily["is_weekend"], "revenue"]
print(f"weekend days: {len(weekend)}, mean ${weekend.mean():,.0f}")
print(f"weekday days: {len(weekday)}, mean ${weekday.mean():,.0f}")
diff = weekend.mean() - weekday.mean()
print(f"observed difference: ${diff:,.0f} per day")

## Welch's t-test
Asks: if both groups came from the same distribution, how likely is a gap this large? Welch's version doesn't assume equal variances.

In [ ]:
t, p = stats.ttest_ind(weekend, weekday, equal_var=False)
pooled_sd = np.sqrt((weekend.var(ddof=1) + weekday.var(ddof=1)) / 2)
print(f"t = {t:.2f}, p = {p:.2g}, Cohen's d (effect size) = {diff / pooled_sd:.2f}")

**Reading it:** a small *p* says the gap is unlikely to be luck. It does **not** say the gap is large or important. Effect size and the dollar difference do that. And a p-value is not the probability that your hypothesis is true.

## The pseudo-replication trap
Run the same test on individual *records* instead of days and watch the p-value become absurdly small. The test thinks you have 3,650 independent observations, which you don't.

In [ ]:
rec_t, rec_p = stats.ttest_ind(df.loc[df["is_weekend"], "revenue"], df.loc[~df["is_weekend"], "revenue"], equal_var=False)
print(f"p on days:    {p:.3g}\np on records: {rec_p:.3g}   <- overconfident")

## Your turn 1: a permutation test
No formulas needed: shuffle the weekend/weekday labels many times and count how often a gap at least as large (in absolute value) appears by chance. Use `rng = np.random.default_rng(0)`, 5,000 shuffles of the **daily** values, and store the two-sided p-value in `p_perm`.

Hint: `values = np.concatenate([weekend, weekday])`, shuffle, take the first `len(weekend)` as the fake weekend.

In [ ]:
p_perm = None

In [ ]:
if p_perm is None:
    print("(write your permutation test above first)")
else:
    check("p_perm is close to the t-test p-value (within 0.02)", abs(p_perm - p) < 0.02, "p_perm = (abs(diffs) >= abs(diff)).mean()")

## Your turn 2: a bootstrap confidence interval
Resample each group **with replacement** 5,000 times (`rng = np.random.default_rng(0)`), compute the difference in means each time, and take the 2.5th and 97.5th percentiles. Store them in `ci_low` and `ci_high`.

In [ ]:
ci_low = None
ci_high = None

In [ ]:
if ci_low is None or ci_high is None:
    print("(write your bootstrap above first)")
else:
    check("interval is ordered and contains the observed difference", ci_low < diff < ci_high)
    print(f"95% CI for the weekend premium: ${ci_low:,.0f} to ${ci_high:,.0f} per day")
    check("agrees with a quick analytic interval (within 25%)", abs((ci_high - ci_low) - 2 * 1.96 * np.sqrt(weekend.var(ddof=1) / len(weekend) + weekday.var(ddof=1) / len(weekday))) / (ci_high - ci_low) < 0.25)

**Discuss:** if the interval is far from zero you can report "weekends earn about $X more per day (95% CI ...)". That sentence is more useful to a manager than any p-value. Also: you ran *one* test. Run twenty and one will look "significant" by chance (multiple comparisons).